In [ ]:
# This script takes the aligned homologs from blastp and deletes suplicate homologs so that each homolog appears exaclty once

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
import json
from importlib import reload
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)



In [ ]:
###### MULTIPLE FILE IMPORT
import src.blast_parser as blast_parser

folder = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/blastp"

df_pos_raw_json = blast_parser.load_blast_group(folder, "fullseq_pos_alignment_primata_1.json", "fullseq_pos_alignment_primata_2.json", "fullseq_pos_alignment_primata_3.json")
df_neg_raw_json = blast_parser.load_blast_group(folder, "fullseq_neg_alignment_primata_1.json",  "fullseq_neg_alignment_primata_2.json")


In [ ]:
print(df_pos_raw_json.head())
print(df_pos_raw_json.shape)

In [ ]:
# Import the functions
from os import path

from src.clean_up_blast_tools.expand import expand_hits_to_motifs
from src.clean_up_blast_tools.coverage import compute_rg_window_columns
from src.clean_up_blast_tools.assignment import assign_shared_accessions

path_pos= "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/pos_RG_regions_win5_metadata.pkl"
path_neg= "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/neg_RG_regions_win5_metadata.pkl"

df_windows_pos = pd.read_pickle(path_pos)
df_windows_neg = pd.read_pickle(path_neg)

df_pos_expanded = expand_hits_to_motifs(df_pos_raw_json, df_windows_pos)
df_neg_expanded = expand_hits_to_motifs(df_neg_raw_json, df_windows_neg)

df_pos_expanded_categorized = compute_rg_window_columns(df_pos_expanded)
df_neg_expanded_categorized = compute_rg_window_columns(df_neg_expanded)

In [ ]:
df_clean_pos = (
    df_pos_expanded_categorized.sort_values("bit_score", ascending=False)
    .drop_duplicates(subset=["hit_accession", "UniqueID", "orig_motif_index"])
    .reset_index(drop=True)
)
df_clean_pos

df_clean_neg = (
    df_neg_expanded_categorized.sort_values("bit_score", ascending=False)
    .drop_duplicates(subset=["hit_accession", "UniqueID", "orig_motif_index"])
    .reset_index(drop=True)
)
df_clean_neg

In [ ]:
def print_assignment_report(report: dict) -> None:
    print("=" * 60)
    print("ACCESSION ASSIGNMENT REPORT")
    print("=" * 60)
    print(f"  Total unique accessions      : {report['n_total_accessions']}")
    print(f"  Shared across >1 UniqueID    : {report['n_shared_accessions']}")
    print(f"  Randomly assigned (ties)     : {report['n_randomly_assigned']}"
          f"  ({report['pct_randomly_assigned']}%)")
    print(f"  Rows before assignment       : {report['n_rows_before']}")
    print(f"  Rows after assignment        : {report['n_rows_after']}")
    print(f"  Rows removed                 : {report['n_rows_removed']}")
    print("=" * 60)


In [ ]:
df_full_pos = df_clean_pos[
    (df_clean_pos["rg_window_coverage"] == "full")].copy()

df_full_pos_final, report = assign_shared_accessions(df_full_pos)

print_assignment_report(report)

df_full_neg = df_clean_neg[
    (df_clean_neg["rg_window_coverage"] == "full")].copy()

df_full_neg_final, report = assign_shared_accessions(df_full_neg)
print_assignment_report(report)

In [ ]:
# Overview statistics of homologs per region
pos_summary = df_full_pos_final.groupby("query_title").agg(
    n_homologs=("hit_accession", "nunique"),
    n_unique_species=("species", "nunique")
).reset_index()
pos_summary["group"] = "positive"
pos_summary["species_per_homolog"] = pos_summary["n_unique_species"] / pos_summary["n_homologs"]

neg_summary = df_full_neg_final.groupby("query_title").agg(
    n_homologs=("hit_accession", "nunique"),
    n_unique_species=("species", "nunique")
).reset_index()
neg_summary["group"] = "negative"
neg_summary["species_per_homolog"] = neg_summary["n_unique_species"] / neg_summary["n_homologs"]

combined_summary = pd.concat([pos_summary, neg_summary], ignore_index=True)

print(f"Positive regions: n = {len(pos_summary)}")
print(f"Negative regions: n = {len(neg_summary)}")

# --- Mann-Whitney U tests (non-parametric, right-skewed count data) ---
from scipy.stats import mannwhitneyu

_, p_hom = mannwhitneyu(
    pos_summary["n_homologs"], neg_summary["n_homologs"], alternative="two-sided"
)
_, p_sp = mannwhitneyu(
    pos_summary["n_unique_species"], neg_summary["n_unique_species"], alternative="two-sided"
)
_, p_ratio = mannwhitneyu(
    pos_summary["species_per_homolog"], neg_summary["species_per_homolog"], alternative="two-sided"
)

# --- plot ---
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))

metrics = [
    ("n_homologs",          "Homologs per region",        "count",   True,  p_hom),
    ("n_unique_species",    "Unique species per region",  "species", True,  p_sp),
    ("species_per_homolog", "Species / homolog ratio",    "ratio",   False, p_ratio),
]
colors = {"positive": "#4daf4a", "negative": "#e41a1c"}

for ax, (metric, title, ylab, log_scale, p_val) in zip(axes, metrics):
    data = [pos_summary[metric].values, neg_summary[metric].values]

    bp = ax.boxplot(
        data, labels=["positive", "negative"],
        widths=0.5, patch_artist=True, showfliers=False,
        medianprops=dict(color="black", linewidth=1.5),
    )
    for patch, grp in zip(bp["boxes"], ["positive", "negative"]):
        patch.set_facecolor(colors[grp])
        patch.set_alpha(0.5)

    for i, (vals, grp) in enumerate(zip(data, ["positive", "negative"]), start=1):
        x_jitter = np.random.normal(i, 0.06, size=len(vals))
        ax.scatter(x_jitter, vals, alpha=0.35, s=10,
                   color=colors[grp], edgecolor="none")

    ax.set_title(title)
    ax.set_ylabel(ylab)
    if log_scale:
        ax.set_yscale("log")
    ax.grid(axis="y", linestyle=":", alpha=0.5)

    ax.text(0.5, 0.97, f"Mann-Whitney p = {p_val:.2e}",
            transform=ax.transAxes, ha="center", va="top",
            fontsize=9, bbox=dict(boxstyle="round,pad=0.3",
                                  facecolor="white", edgecolor="lightgray"))

fig.suptitle("Homolog recruitment: positive vs negative regions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.clean_up_blast_tools.sequence_extraction import extract_rg_region_seqs

# --- label, extract, combine ---
df_full_pos_final["group"] = "positive"
df_full_neg_final["group"] = "negative"

df_full_pos_final = extract_rg_region_seqs(df_full_pos_final)
df_full_neg_final = extract_rg_region_seqs(df_full_neg_final)

df_combined = pd.concat([df_full_pos_final, df_full_neg_final], ignore_index=True)

# --- sanity check: one row per query_id should have consistent window lengths ---
uniq_per_motif = (
    df_combined.groupby(["query_id", "orig_motif_index"])[
        ["left_flank", "motif_length", "right_flank"]
    ]
    .nunique()
    .max()
)
if (uniq_per_motif > 1).any():
    print("WARNING: some (query_id, orig_motif_index) pairs have varying "
          "flank/motif lengths across rows:")
    print(uniq_per_motif)

# --- exploratory plot: window region size distributions, split by group ---
df_dedup = df_combined.drop_duplicates(subset=["query_id", "orig_motif_index"])

colors = {"positive": "#4daf4a", "negative": "#e41a1c"}
components = ["left_flank", "motif_length", "right_flank"]
component_labels = ["Left flank", "Motif", "Right flank"]
groups = ["positive", "negative"]

fig, ax = plt.subplots(figsize=(8, 4.5))

positions = []
box_data = []
box_colors = []
for i, comp in enumerate(components):
    for j, grp in enumerate(groups):
        # offset within each component cluster: -0.2 for pos, +0.2 for neg
        positions.append(i + 1 + (j - 0.5) * 0.4)
        box_data.append(df_dedup.loc[df_dedup["group"] == grp, comp].dropna().values)
        box_colors.append(colors[grp])

bp = ax.boxplot(
    box_data, positions=positions, widths=0.35,
    patch_artist=True, showfliers=False,
    medianprops=dict(color="black", linewidth=1.5),
)
for patch, c in zip(bp["boxes"], box_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.5)

# jittered points
for pos, vals, c in zip(positions, box_data, box_colors):
    x_jitter = np.random.normal(pos, 0.04, size=len(vals))
    ax.scatter(x_jitter, vals, alpha=0.35, s=10, color=c, edgecolor="none")

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(component_labels)
ax.set_ylabel("Length (aa)")
ax.set_title("Window region size distributions (pos vs neg)")
ax.grid(axis="y", linestyle=":", alpha=0.5)

# manual legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=colors[g], alpha=0.5, label=g) for g in groups]
ax.legend(handles=legend_handles, loc="upper right", frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# --- save as final step ---
output_path = Path(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/combined_blast_results.pkl"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
df_combined.to_pickle(output_path)
print(f"Saved combined dataframe to {output_path}")
print(f"Total rows: {len(df_combined)}")
print(f"Group counts:\n{df_combined['group'].value_counts()}")